## Annotation nombre de termes

## 1. Import des librairies

In [1]:
import pandas as pd
import numpy as np
import math
import re
from collections import Counter
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk import bigrams
from nltk import trigrams
from gensim.models import FastText
import ast

c:\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.3.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


## 2. Import des données

In [2]:
df = pd.read_csv('voies_par_longueur_annotation_cotationfr.csv')

In [3]:
df

,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,difficulty,height_corrigee,height_variante,height_approx,difficulty_note,difficulty_extra,equipment,systeme_cotation,composition,log_nb_terms_description
0,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,5c,False,False,False,NaN,NaN,NaN,francaise,NaN,3.401197
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,5c,False,False,False,NaN,NaN,NaN,francaise,NaN,2.484907
2,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,5c+,False,False,False,NaN,NaN,NaN,francaise,NaN,3.178054
3,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,5b+,False,False,False,NaN,NaN,NaN,francaise,NaN,3.218876
4,1944609,Capricho,Aiguille de la Canalona,medium,rock_climbing,multi,D+,NaN,NaN,P2+,...,5c,False,False,False,NaN,NaN,NaN,francaise,NaN,4.290459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42856,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,4c,False,False,False,NaN,NaN,NaN,francaise,NaN,1.945910
42857,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,4a,False,False,False,NaN,NaN,NaN,francaise,NaN,4.127134
42858,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,5b,False,False,False,NaN,NaN,NaN,francaise,NaN,2.564949
42859,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,5b,False,False,False,NaN,NaN,NaN,francaise,NaN,3.178054


## 3. Extraction des éléments naturels

In [ ]:
# Etablissement du dictionnaire des termes qu'on veut récupérer dans les descriptions 

dico_terme_description = {
    'dévers': ['dévers', 'déversante', 'déversant', 'déversantes', 'déversants', 'déversé', 'déversée', 'déversées', 'déversés'],
    'surplomb': ['surplombant', 'surplombante', 'surplombé', 'surplomb', 'surplombs',
                  'bombé', 'bombés', 'bombée', 'bombées', 'toit', 'toits'],
    'dalle': ['dalle', 'dalles', 'adhérence', 'adhérences'],
    'fissure': ['fissure', 'fissures', 'faille', 'failles'],
    'épaule': ['épaule', 'épaules', 'épaulement', 'épaulements'],
    'dièdre': ['dièdre', 'dièdres'],
    'cheminée': ['cheminée', 'cheminées'],
    'vire': ['vire', 'vires', 'virette', 'virettes'],
    'terrasse': ['terrasse', 'terrasses', 'plateforme', 'plateformes'],
    'traversée': ['traversée', 'traversées', 'traverser', 'traverse', 'traversant'],
    'réglette': ['réglette', 'réglettes'],
    'bac': ['bac', 'grosse prise', 'grosses prises', 'bonne prise', 'bonnes prises'],
    'plat': ['plat', 'plats',
             'replat', 'replats'],
    'pince': ['pince', 'pinces'],
    'mono_doigt': ['mono-doigt', 'monodoigt', 'mono-doigts', 'monodoigts'],
    'bi_doigt': ['bi-doigt', 'bidoigt', 'bi-doigts', 'bidoigts'],
    'inversee': ['inversée', 'inversées', 'inversé', 'inversés', 'inverse', 'inverses'],
    'crochet': ['crochet', 'crochets'],
    'trou': ['trou', 'trous'],
    'goutte_eau': ["goutte d'eau", "gouttes d'eau"],
    'petite_prise': ['petite prise', 'petites prises'],
    'écailles': ['écaille', 'écailles'],
    'arête': ['arête', 'arêtes'],
    'pilier': ['pilier', 'piliers'],
    'couloir': ['couloir', 'couloirs'],
    'cannelure': ['cannelure', 'cannelures'],
    'éperon': ['éperon', 'éperons'],
    'contrefort': ['contrefort', 'contreforts'],
    'corniche': ['corniche', 'corniches'],
    'fissure-dièdre': ['fissure-dièdre', 'fissures-dièdres', 'fissure dièdre', 'fissures dièdres', 'fissures dièdre'],
}

In [4]:
# Fonction qui compile une regex par catégorie à récupérer

def compiler_dictionnaire(dico):
    """
    Compile un regex par catégorie du dictionnaire.
    - re.escape(t) : traite chaque terme comme du texte littéral (important
      pour les termes avec apostrophe ou tiret, ex: "goutte d'eau", "mono-doigt",
      qui contiennent des caractères ayant un sens spécial en regex sans escape).
    - \\b ... \\b : "word boundary", garantit qu'on détecte le mot entier, pas
      une simple sous-chaîne à l'intérieur d'un autre mot.
    - re.IGNORECASE : insensible à la casse (DÉVERS, Dévers, dévers... tous détectés).
    """
    regex_par_categorie = {}
    for categorie, termes in dico.items():
        termes_tries = sorted(termes, key=len, reverse=True)
        motif = r'\b(?:' + '|'.join(re.escape(t) for t in termes_tries) + r')\b'
        regex_par_categorie[categorie] = re.compile(motif, re.IGNORECASE)
    return regex_par_categorie


# Renvoie un dict {catégorie: liste des occurrences trouvées} pour un texte.

def detecter_termes(texte, regex_dict):
    if not isinstance(texte, str):
        return {}
    resultats = {}
    for categorie, pattern in regex_dict.items():
        matches = pattern.findall(texte)
        if matches:
            resultats[categorie] = matches
    return resultats


regex_dict = compiler_dictionnaire(dico_terme_description)

In [5]:
df_annotation = df.copy()

# Calcul du NOMBRE d'occurrences plutôt qu'un simple booléen :
for categorie in dico_terme_description:
    df_annotation[f'nb_{categorie}'] = df_annotation['description'].apply(
        lambda texte: len(detecter_termes(texte, regex_dict).get(categorie, []))
    )

# Calcul de la LISTE des occurrences trouvées pour chaque catégorie :
for categorie in dico_terme_description:
    df_annotation[categorie] = df_annotation['description'].apply(
        lambda texte: detecter_termes(texte, regex_dict).get(categorie, [])
    )

## 4. Extraction des séquences d'éléments naturels

In [6]:
# Création d'une variable qui contient la liste des catégories détectées dans l'ordre d'apparition dans le texte.
def detecter_termes_ordonnes(texte, dico):
    if not isinstance(texte, str):
        return []

    occurrences = []

    for categorie, termes in dico.items():
        for terme in termes:
            motif = re.compile(
                r'\b' + re.escape(terme) + r'\b',
                re.IGNORECASE
            )

            for match in motif.finditer(texte):
                occurrences.append((match.start(), categorie))

    occurrences.sort(key=lambda x: x[0])

    return [categorie for _, categorie in occurrences]


# Ajout de la colonne sans modifier les colonnes existantes
df_annotation['termes_ordonnes'] = (
    df_annotation['description'].apply(
        lambda texte: detecter_termes_ordonnes(
            texte, dico_terme_description
        )
    )
)

In [7]:
# Création des variables de transition d'éléments (bigrammes)

seq_col = "termes_ordonnes"  
def vers_liste(s):
    if isinstance(s, list):
        return s
    if pd.isna(s):
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [s]
    except (ValueError, SyntaxError):
        return [s]

seqs = df_annotation[seq_col].apply(vers_liste)

# --- 1. Distribution des longueurs ---
longueurs = seqs.apply(len)
print(longueurs.describe())
print("\nRépartition des longueurs :")
print(longueurs.value_counts().sort_index().head(15))
n2 = (longueurs >= 2).sum()
print(f"\nSéquences avec >= 2 éléments (donc au moins une transition) : "
      f"{n2} ({100 * n2 / len(seqs):.1f} %)")

# --- 2. Vocabulaire des éléments ---
vocab = Counter(e for s in seqs for e in s)
print(f"\n{len(vocab)} éléments distincts ; top 30 :")
print(vocab.most_common(30))

# --- 3. Transitions (bigrammes) ---
bigrams = Counter()
for s in seqs:
    for a, b in zip(s, s[1:]):
        bigrams[(a, b)] += 1
print(f"\n{len(bigrams)} transitions distinctes ; top 30 :")
for (a, b), n in bigrams.most_common(30):
    print(f"  {a} -> {b} : {n}")

count    42861.000000
mean         1.599473
std          1.608853
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         24.000000
Name: termes_ordonnes, dtype: float64

Répartition des longueurs :
termes_ordonnes
0     11302
1     13456
2      8694
3      4771
4      2326
5      1122
6       592
7       267
8       154
9        83
10       49
11       16
12       10
13        9
14        2
Name: count, dtype: int64

Séquences avec >= 2 éléments (donc au moins une transition) : 18103 (42.2 %)

30 éléments distincts ; top 30 :
[('dalle', 11767), ('fissure', 9579), ('traversée', 9541), ('dièdre', 8161), ('surplomb', 6074), ('vire', 4977), ('pilier', 2321), ('cheminée', 2203), ('terrasse', 2099), ('dévers', 1835), ('arête', 1737), ('écailles', 1598), ('bac', 1027), ('éperon', 926), ('couloir', 911), ('trou', 793), ('cannelure', 654), ('réglette', 401), ('goutte_eau', 367), ('fissure-dièdre', 359), ('crochet', 320), ('petite_prise', 288), ('plat

## 5. Analyse sur les lignes vides pour voir si certains termes ont été oubliés

### 5.1 Mots fréquents dans les descriptions où l'extraction de termes n'a rien donné

In [8]:
# Lignes où toutes les catégories sont vides mais la description n'est pas vide

categories = list(dico_terme_description.keys())

def est_vide(valeur):
    if isinstance(valeur, list):
        return len(valeur) == 0
    return pd.isna(valeur)

toutes_vides = df_annotation[categories].applymap(est_vide).all(axis=1)
description_non_vide = df_annotation['description'].notna() & (df_annotation['description'].str.strip() != "")

lignes_a_investiguer = df_annotation[toutes_vides & description_non_vide]
print(f"Nombre de lignes à investiguer : {len(lignes_a_investiguer)}")

C:\Users\lunag\AppData\Local\Temp\ipykernel_3592\3576918466.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  toutes_vides = df_annotation[categories].applymap(est_vide).all(axis=1)


Nombre de lignes à investiguer : 11302


In [9]:
# Monogrammes

mots_vides = set(stopwords.words('french'))
mots_deja_connus = set()
for termes in dico_terme_description.values():
    for t in termes:
        mots_deja_connus.update(t.lower().split())

def tokeniser(texte):
    return re.findall(r"[a-zàâäéèêëïîôöùûüç]+", texte.lower())

compteur = Counter()
for texte in lignes_a_investiguer['description'].dropna():
    for mot in tokeniser(texte):
        if mot not in mots_vides and mot not in mots_deja_connus and len(mot) > 2:
            compteur[mot] += 1

for mot, freq in compteur.most_common(50):
    print(f"{mot:20s} {freq}")

relais               2075
longueur             2015
puis                 1951
gauche               1933
droite               1905
droit                1362
mur                  1196
plus                 1091
très                 958
départ               942
peu                  911
rocher               880
facile               859
bien                 853
raide                825
tout                 786
dessus               757
jusqu                684
ascendance           664
vers                 586
remonter             549
fin                  533
points               508
belle                502
relai                496
sommet               494
spits                461
voie                 441
rejoindre            435
sup                  398
bloc                 386
courte               359
deux                 353
quelques             351
sortie               343
assez                342
beau                 338
technique            317
point                315
spit             

In [12]:
# Bigrammes

compteur_bigrammes = Counter()
for texte in lignes_a_investiguer['description'].dropna():
    mots = [m for m in tokeniser(texte) if m not in mots_vides]
    for bg in bigrams(mots):
        compteur_bigrammes[' '.join(bg)] += 1

for bg, freq in compteur_bigrammes.most_common(50):
    print(f"{bg:30s} {freq}")

tout droit                     661
droit dessus                   394
mur raide                      317
belle longueur                 309
ascendance gauche              277
ascendance droite              274
jusqu relais                   245
vers gauche                    241
plus facile                    241
relais r                       214
vers droite                    203
droite puis                    178
gauche puis                    171
courte longueur                156
dessus relais                  142
départ puis                    132
très belle                     129
puis tout                      126
jusqu sommet                   116
droit puis                     114
petit mur                      104
beau mur                       104
remonter mur                   103
rejoindre relais               102
quelques mètres                101
puis plus                      99
sup e                          97
e sup                          97
peu plus               

In [13]:
# Trigrammes

compteur_trigrammes = Counter()
for texte in lignes_a_investiguer['description'].dropna():
    mots = [m for m in tokeniser(texte) if m not in mots_vides]
    for bg in trigrams(mots):
        compteur_trigrammes[' '.join(bg)] += 1

for bg, freq in compteur_trigrammes.most_common(50):
    print(f"{bg:30s} {freq}")

puis tout droit                123
jusqu relais r                 99
sup e sup                      97
très belle longueur            86
sup er sup                     76
puis plus facile               75
droit dessus relais            69
tout droit puis                65
relais r points                54
ascendance vers gauche         53
rejoindre relais r             53
points non reliés              53
gauche puis tout               46
er sup point                   38
r points non                   38
ascendance vers droite         37
droit dessus puis              36
droite puis tout               35
légère ascendance gauche       35
droite puis droit              35
fino alla sosta                35
e sup point                    34
relais rappel r                34
remonter mur raide             33
ascendance gauche puis         30
tout droit jusqu               29
relais chaîné r                29
ascendance droite jusqu        28
puis droit dessus              27
puis ascendan

### 5.2 Tokenisation

In [14]:
# --- Étape 1 : construire les règles de fusion pour les termes composés ---
def construire_motifs_fusion(dico):
    remplacements = []
    for termes in dico.values():
        for t in termes:
            if ' ' in t or "-" in t or "'" in t:
                token_fusionne = re.sub(r"[ '\-]+", "_", t.lower())
                remplacements.append((t, token_fusionne))
    remplacements.sort(key=lambda x: len(x[0]), reverse=True)
    return remplacements

remplacements = construire_motifs_fusion(dico_terme_description)

def preparer_texte(texte):
    """Fusionne les termes composés puis tokenise le texte."""
    if not isinstance(texte, str):
        return []
    texte = texte.lower()
    for original, fusionne in remplacements:
        texte = re.sub(re.escape(original.lower()), fusionne, texte)
    return re.findall(r"[a-zàâäéèêëïîôöùûüç_]+", texte)

# --- Étape 2 : construire le corpus d'entraînement ---
# On utilise TOUTES les descriptions non vides (pas seulement les "vides à investiguer"),
# pour donner au modèle le maximum de contexte possible
sentences = [
    preparer_texte(texte)
    for texte in df_annotation['description'].dropna()
    if isinstance(texte, str) and texte.strip() != ""
]
sentences = [s for s in sentences if len(s) > 0]
print(f"Nombre de phrases dans le corpus d'entraînement : {len(sentences)}")

# --- Étape 3 : entraîner le modèle ---
model = FastText(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,     # ignore les mots apparaissant moins de 5 fois (bruit)
    sg=1,
    epochs=15,
    workers=4,
)

# --- Étape 4 : interroger les voisins de chaque terme de ton dictionnaire ---
mots_deja_connus = set()
for termes in dico_terme_description.values():
    for t in termes:
        token = re.sub(r"[ '\-]+", "_", t.lower()) if (' ' in t or '-' in t or "'" in t) else t.lower()
        mots_deja_connus.add(token)

for categorie, termes in dico_terme_description.items():
    for terme in termes:
        token = re.sub(r"[ '\-]+", "_", terme.lower()) if (' ' in terme or '-' in terme or "'" in terme) else terme.lower()
        if token not in model.wv:
            continue
        voisins = model.wv.most_similar(token, topn=10)
        # On ne garde que les voisins qui ne sont PAS déjà dans le dictionnaire
        nouveaux_candidats = [(v, s) for v, s in voisins if v not in mots_deja_connus]
        if nouveaux_candidats:
            print(f"\n[{categorie}] voisins de '{terme}' :")
            for v, s in nouveaux_candidats:
                print(f"    {v:20s} {s:.3f}")

Nombre de phrases dans le corpus d'entraînement : 42811

[dévers] voisins de 'dévers' :
    déverse              0.673
    bombements           0.644
    surplombe            0.641

[dévers] voisins de 'déversante' :
    traversante          0.712
    patinée              0.646
    plaisante            0.633
    déverse              0.621
    fatigante            0.619

[dévers] voisins de 'déversant' :
    surplombants         0.681
    raide                0.673
    déverse              0.650

[dévers] voisins de 'déversantes' :
    surplombantes        0.841
    rondes               0.755
    diverses             0.734
    surplombants         0.733
    patinées             0.724
    riches               0.722
    arquées              0.714
    franches             0.706

[dévers] voisins de 'déversants' :
    surplombants         0.847
    pnts                 0.716
    surplombantes        0.707
    gants                0.691
    verticaux            0.684
    fuyants             

## 6. Enregistrement du dataset

In [15]:
df_annotation.to_csv("annotation_compte_mots.csv", index=False, encoding="utf-8")